In [45]:
import pandas as pd
import os

In [46]:
%run analysis_utils.py

In [47]:
# base_dir = '/home/haitham/mnt/__output_v4/exp'
# base_dir = '/home/haitham/mnt/__output_v3/exp'


# search_text = 'cls_metrics_vote'
# folders= [] 
# # Use os.walk to search through all subdirectories
# for root, dirs, files in os.walk(base_dir):
#     for directory in dirs:
#         # print(directory)
#         if search_text in directory:
#             # Print the full path of matching folders
#             print(os.path.join(root, directory))
#             folders.append(os.path.join(root, directory))
from pathlib import Path

def find_files(root_folder, filename):
    """
    Search for all instances of a file inside a folder and its subfolders.

    Parameters
    ----------
    root_folder : str or Path
        Folder to search inside.
    filename : str
        Exact file name to search for, e.g. "data.h5ad".

    Returns
    -------
    list[Path]
        List of matching file paths.
    """
    root_folder = Path(root_folder)

    if not root_folder.exists():
        raise FileNotFoundError(f"Folder does not exist: {root_folder}")

    matches = list(root_folder.rglob(filename))
    return matches


# Example usage
root = '/home/haitham/mnt/__output_v4/exp'
# root = '/home/haitham/mnt/__output_v3/exp'


vote_files = find_files(root, 'vote_cv_metrics.csv')
avg_files = find_files(root, 'avg_cv_metrics.csv')
mil_files = find_files(root, 'mil_cv_metrics.csv')

# print(f"Found {len(matches)} matching file(s):")
# for path in matches:
#     print(path)

In [48]:
model_name_map={
'hvg': 'HVG',
'pca': 'PCA',
'scgpt': 'scGPT', 
'scgpt_cancer': 'scGPT [cancer]',
'scvi':'scVI',
'scvi_donor_id':'scVI',
'scfoundation':'scFoundation',
'scimilarity':'SCimiarity',
'cellplm':'CellPLM',
'gf-6L-30M-i2048': 'GF-V1',
'gf-6L-30M-i2048_continue': 'GF-V1 [continue]',
'Geneformer-V2-104M_CLcancer': 'GF-V2 [cancer]',
'Geneformer-V2-104M': 'GF-V2',
'Geneformer-V2-104M_continue': 'GF-V2 [continue]',
'Geneformer-V2-316M': 'GF-V2-Deep',
'gf-6L-30M-i2048_finetune': 'GF-V1 [finetune]',
'Geneformer-V2-104M_finetune': 'GF-V2 [finetune]',
'hvg_seurat_4096': 'HVG' ,                        
'state_se600m_epoch16': 'STATE',                     
'scfoundation_brca_cancer_cells': 'scFoundation',         
'geneformer_V2-104M_CLcancer-i4096':'GF-V2 [cancer]',
'geneformer_V2-316M-i4096':'GF-V2-Deep'                    , 
'continue_geneformer_V1-10M-i2048_continue':'GF-V1 [continue]',     
'geneformer_V1-10M-i2048':'GF-V1'       ,
'continue_geneformer_V2-104M-i4096_continue': 'GF-V2 [continue]'  ,  
'geneformer_V2-104M-i4096': 'GF-V2'                 ,
'scgpt_cancer-i2048' :'scGPT [cancer]',                           
'scgpt_human-i2048'  : 'scGPT',                          
'cellplm_85M-20231027'  :'CellPLM',                       
'scimilarity_v1.1': 'SCimiarity'                              ,
'pca_n100'  :'PCA [100]'  ,
'pca_n50'   : 'PCA [50]',                                   
'pca_n20'                : 'PCA [20]'        ,               
'scvi'                               :'scVI'     ,      
'scconcept_corpus30m' : 'scConcept'                          ,
'nicheformer_nicheformer'                       :'Nicheformer',


}

In [49]:
df = pd.read_csv(vote_files[0])
df.groupby('Metrics')['randomforest'].mean().to_frame().T

Metrics,AUC,AUPRC,Accuracy,F1,Precision,Recall
randomforest,0.930159,0.951005,0.757692,0.754319,0.777024,0.761905


In [50]:
df.pivot(index='Metrics', columns='fold', values='randomforest').T

Metrics,AUC,AUPRC,Accuracy,F1,Precision,Recall
fold,,,,,,
fold_1,0.952381,0.961735,0.769231,0.763636,0.833333,0.785714
fold_2,0.976190,0.976190,0.769231,0.769231,0.773810,0.773810
fold_3,0.861111,0.924242,0.666667,0.657143,0.687500,0.666667
fold_4,0.972222,0.976190,0.833333,0.833333,0.833333,0.833333
fold_5,0.888889,0.916667,0.750000,0.748252,0.757143,0.750000


In [51]:
df.set_index('Metrics')

,randomforest,fold
Metrics,,
AUC,0.952381,fold_1
AUPRC,0.961735,fold_1
F1,0.763636,fold_1
Accuracy,0.769231,fold_1
Precision,0.833333,fold_1
Recall,0.785714,fold_1
AUC,0.976190,fold_2
AUPRC,0.976190,fold_2
F1,0.769231,fold_2


In [52]:
import json
vote_dfs = []
for matched_file in vote_files:
    df = pd.read_csv(matched_file)

    # df = df.T
    df = df.pivot(index='Metrics', columns='fold', values='randomforest').T
    # df = df.groupby('Metrics')['randomforest'].mean().to_frame().T
    run_summary_path = matched_file.parent.parent / "run_summary.json"
    with open(run_summary_path, "r") as f:
            run_summary = json.load(f)
            model = run_summary['run_id']
            exp = Path(run_summary['config_path']).stem
            model  = model.replace(f'_{exp}', "")
        
    df['model'] = model
    df['exp'] = exp
    df['strategy'] = 'vote'
    vote_dfs.append(df)
        
    

In [53]:
vote_dfs[0]

Metrics,AUC,AUPRC,Accuracy,F1,Precision,Recall,model,exp,strategy
fold,,,,,,,,,
fold_1,0.952381,0.961735,0.769231,0.763636,0.833333,0.785714,geneformer_V1-10M-i2048_brca_pre_post,brca_pre_post_counts,vote
fold_2,0.976190,0.976190,0.769231,0.769231,0.773810,0.773810,geneformer_V1-10M-i2048_brca_pre_post,brca_pre_post_counts,vote
fold_3,0.861111,0.924242,0.666667,0.657143,0.687500,0.666667,geneformer_V1-10M-i2048_brca_pre_post,brca_pre_post_counts,vote
fold_4,0.972222,0.976190,0.833333,0.833333,0.833333,0.833333,geneformer_V1-10M-i2048_brca_pre_post,brca_pre_post_counts,vote
fold_5,0.888889,0.916667,0.750000,0.748252,0.757143,0.750000,geneformer_V1-10M-i2048_brca_pre_post,brca_pre_post_counts,vote


In [54]:
vote_df = pd.concat(vote_dfs)
vote_df = vote_df.reset_index().drop('fold', axis=1)
vote_df

Metrics,AUC,AUPRC,Accuracy,F1,Precision,Recall,model,exp,strategy
0,0.952381,0.961735,0.769231,0.763636,0.833333,0.785714,geneformer_V1-10M-i2048_brca_pre_post,brca_pre_post_counts,vote
1,0.976190,0.976190,0.769231,0.769231,0.773810,0.773810,geneformer_V1-10M-i2048_brca_pre_post,brca_pre_post_counts,vote
2,0.861111,0.924242,0.666667,0.657143,0.687500,0.666667,geneformer_V1-10M-i2048_brca_pre_post,brca_pre_post_counts,vote
3,0.972222,0.976190,0.833333,0.833333,0.833333,0.833333,geneformer_V1-10M-i2048_brca_pre_post,brca_pre_post_counts,vote
4,0.888889,0.916667,0.750000,0.748252,0.757143,0.750000,geneformer_V1-10M-i2048_brca_pre_post,brca_pre_post_counts,vote
5,1.000000,1.000000,0.833333,0.828571,0.875000,0.833333,geneformer_V1-10M-i2048_brca_subtype,brca_subtype_counts,vote
6,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,geneformer_V1-10M-i2048_brca_subtype,brca_subtype_counts,vote
7,0.777778,0.866667,0.666667,0.666667,0.666667,0.666667,geneformer_V1-10M-i2048_brca_subtype,brca_subtype_counts,vote
8,1.000000,1.000000,0.800000,0.800000,0.833333,0.833333,geneformer_V1-10M-i2048_brca_subtype,brca_subtype_counts,vote
9,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,geneformer_V1-10M-i2048_brca_subtype,brca_subtype_counts,vote


In [55]:
avg_dfs = []
for matched_file in avg_files:
    df = pd.read_csv(matched_file)

    # df = df.T
    # df = df.groupby('Metrics')['randomforest'].mean().to_frame().T
    df = df.pivot(index='Metrics', columns='fold', values='randomforest').T
    run_summary_path = matched_file.parent.parent / "run_summary.json"
    with open(run_summary_path, "r") as f:
            run_summary = json.load(f)
            model = run_summary['run_id']
            exp = Path(run_summary['config_path']).stem
            model  = model.replace(f'_{exp}', "")
        
    df['model'] = model
    df['exp'] = exp
    df['strategy'] = 'avg'
    avg_dfs.append(df)
    

In [56]:
avg_df = pd.concat(avg_dfs)
avg_df = avg_df.reset_index().drop('fold', axis=1)
avg_df

Metrics,AUC,AUPRC,Accuracy,F1,Precision,Recall,model,exp,strategy
0,0.904762,0.948052,0.923077,0.923077,0.928571,0.928571,geneformer_V1-10M-i2048_brca_pre_post,brca_pre_post_counts,avg
1,0.976190,0.976190,0.846154,0.845238,0.875000,0.857143,geneformer_V1-10M-i2048_brca_pre_post,brca_pre_post_counts,avg
2,0.944444,0.958333,0.916667,0.916084,0.928571,0.916667,geneformer_V1-10M-i2048_brca_pre_post,brca_pre_post_counts,avg
3,0.805556,0.775000,0.833333,0.833333,0.833333,0.833333,geneformer_V1-10M-i2048_brca_pre_post,brca_pre_post_counts,avg
4,0.833333,0.881944,0.583333,0.580420,0.585714,0.583333,geneformer_V1-10M-i2048_brca_pre_post,brca_pre_post_counts,avg
5,1.000000,1.000000,0.833333,0.828571,0.875000,0.833333,geneformer_V1-10M-i2048_brca_subtype,brca_subtype_counts,avg
6,0.777778,0.805556,0.833333,0.828571,0.875000,0.833333,geneformer_V1-10M-i2048_brca_subtype,brca_subtype_counts,avg
7,0.666667,0.755556,0.666667,0.666667,0.666667,0.666667,geneformer_V1-10M-i2048_brca_subtype,brca_subtype_counts,avg
8,0.833333,0.833333,0.800000,0.800000,0.833333,0.833333,geneformer_V1-10M-i2048_brca_subtype,brca_subtype_counts,avg
9,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,geneformer_V1-10M-i2048_brca_subtype,brca_subtype_counts,avg


In [57]:
avg_df[avg_df['model'].str.contains('geneformer_V1-10M')& avg_df['exp'].str.contains('brca_pre_post')]

Metrics,AUC,AUPRC,Accuracy,F1,Precision,Recall,model,exp,strategy
0,0.904762,0.948052,0.923077,0.923077,0.928571,0.928571,geneformer_V1-10M-i2048_brca_pre_post,brca_pre_post_counts,avg
1,0.976190,0.976190,0.846154,0.845238,0.875000,0.857143,geneformer_V1-10M-i2048_brca_pre_post,brca_pre_post_counts,avg
2,0.944444,0.958333,0.916667,0.916084,0.928571,0.916667,geneformer_V1-10M-i2048_brca_pre_post,brca_pre_post_counts,avg
3,0.805556,0.775000,0.833333,0.833333,0.833333,0.833333,geneformer_V1-10M-i2048_brca_pre_post,brca_pre_post_counts,avg
4,0.833333,0.881944,0.583333,0.580420,0.585714,0.583333,geneformer_V1-10M-i2048_brca_pre_post,brca_pre_post_counts,avg


In [58]:
mil_dfs = []
for matched_file in mil_files:
    df = pd.read_csv(matched_file)

    # df = df.T
    # df = df.groupby('Metrics')['randomforest'].mean().to_frame().T
    df = df.pivot(index='Metrics', columns='fold', values='randomforest').T
    run_summary_path = matched_file.parent.parent / "run_summary.json"
    with open(run_summary_path, "r") as f:
            run_summary = json.load(f)
            model = run_summary['run_id']
            exp = Path(run_summary['config_path']).stem
            model  = model.replace(f'_{exp}', "")
        
    df['model'] = model
    df['exp'] = exp
    df['strategy'] = 'MIL'
    mil_dfs.append(df)

In [59]:
mil_df = pd.concat(mil_dfs)
mil_df = mil_df.reset_index().drop('fold', axis=1)
mil_df


Metrics,AUC,AUPRC,Accuracy,F1,Precision,Recall,model,exp,strategy
0,0.952381,0.961735,0.769231,0.769231,0.773810,0.773810,geneformer_V1-10M-i2048_brca_pre_post,brca_pre_post_counts,MIL
1,0.904762,0.933333,0.769231,0.769231,0.773810,0.773810,geneformer_V1-10M-i2048_brca_pre_post,brca_pre_post_counts,MIL
2,0.944444,0.958333,0.833333,0.833333,0.833333,0.833333,geneformer_V1-10M-i2048_brca_pre_post,brca_pre_post_counts,MIL
3,0.833333,0.855159,0.666667,0.666667,0.666667,0.666667,geneformer_V1-10M-i2048_brca_pre_post,brca_pre_post_counts,MIL
4,0.750000,0.828409,0.750000,0.733333,0.833333,0.750000,geneformer_V1-10M-i2048_brca_pre_post,brca_pre_post_counts,MIL
5,1.000000,1.000000,0.833333,0.828571,0.875000,0.833333,geneformer_V1-10M-i2048_brca_subtype,brca_subtype_counts,MIL
6,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,geneformer_V1-10M-i2048_brca_subtype,brca_subtype_counts,MIL
7,0.888889,0.916667,0.833333,0.828571,0.875000,0.833333,geneformer_V1-10M-i2048_brca_subtype,brca_subtype_counts,MIL
8,1.000000,1.000000,0.800000,0.761905,0.875000,0.750000,geneformer_V1-10M-i2048_brca_subtype,brca_subtype_counts,MIL
9,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,geneformer_V1-10M-i2048_brca_subtype,brca_subtype_counts,MIL


In [60]:
calssification_metrics = pd.concat([vote_df,mil_df, avg_df ])

In [61]:
calssification_metrics.model.unique()

array(['geneformer_V1-10M-i2048_brca_pre_post',
       'geneformer_V1-10M-i2048_brca_subtype',
       'geneformer_V1-10M-i2048_brca_chemo'], dtype=object)

In [62]:
model_name_map['hvg_seurat_4096']

'HVG'

In [63]:
calssification_metrics['group'] =  calssification_metrics['model'].map(map_groups)
# calssification_metrics['model'] =  calssification_metrics['model'].map(model_name_map)
calssification_metrics

Metrics,AUC,AUPRC,Accuracy,F1,Precision,Recall,model,exp,strategy,group
0,0.952381,0.961735,0.769231,0.763636,0.833333,0.785714,geneformer_V1-10M-i2048_brca_pre_post,brca_pre_post_counts,vote,Geneformer
1,0.976190,0.976190,0.769231,0.769231,0.773810,0.773810,geneformer_V1-10M-i2048_brca_pre_post,brca_pre_post_counts,vote,Geneformer
2,0.861111,0.924242,0.666667,0.657143,0.687500,0.666667,geneformer_V1-10M-i2048_brca_pre_post,brca_pre_post_counts,vote,Geneformer
3,0.972222,0.976190,0.833333,0.833333,0.833333,0.833333,geneformer_V1-10M-i2048_brca_pre_post,brca_pre_post_counts,vote,Geneformer
4,0.888889,0.916667,0.750000,0.748252,0.757143,0.750000,geneformer_V1-10M-i2048_brca_pre_post,brca_pre_post_counts,vote,Geneformer
5,1.000000,1.000000,0.833333,0.828571,0.875000,0.833333,geneformer_V1-10M-i2048_brca_subtype,brca_subtype_counts,vote,Geneformer
6,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,geneformer_V1-10M-i2048_brca_subtype,brca_subtype_counts,vote,Geneformer
7,0.777778,0.866667,0.666667,0.666667,0.666667,0.666667,geneformer_V1-10M-i2048_brca_subtype,brca_subtype_counts,vote,Geneformer
8,1.000000,1.000000,0.800000,0.800000,0.833333,0.833333,geneformer_V1-10M-i2048_brca_subtype,brca_subtype_counts,vote,Geneformer
9,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,geneformer_V1-10M-i2048_brca_subtype,brca_subtype_counts,vote,Geneformer


In [64]:
calssification_metrics.to_csv('./results/classification_metrcis.csv')

In [65]:
calssification_metrics[['model', 'exp']]

Metrics,model,exp
0,geneformer_V1-10M-i2048_brca_pre_post,brca_pre_post_counts
1,geneformer_V1-10M-i2048_brca_pre_post,brca_pre_post_counts
2,geneformer_V1-10M-i2048_brca_pre_post,brca_pre_post_counts
3,geneformer_V1-10M-i2048_brca_pre_post,brca_pre_post_counts
4,geneformer_V1-10M-i2048_brca_pre_post,brca_pre_post_counts
5,geneformer_V1-10M-i2048_brca_subtype,brca_subtype_counts
6,geneformer_V1-10M-i2048_brca_subtype,brca_subtype_counts
7,geneformer_V1-10M-i2048_brca_subtype,brca_subtype_counts
8,geneformer_V1-10M-i2048_brca_subtype,brca_subtype_counts
9,geneformer_V1-10M-i2048_brca_subtype,brca_subtype_counts


In [66]:
# def extract_model_name(filename: str, marker: str) -> str:
#     """
#     Extract the text between two occurrences of `marker`.

#     Example:
#     filename = "brca_cell_type_hvg_seurat_4096_brca_cell_type"
#     marker = "brca_cell_type"
#     returns: "hvg_seurat_4096"
#     """
#     parts = filename.split(marker)

#     if len(parts) < 3:
#         raise ValueError(f"Marker '{marker}' must appear at least twice.")

#     model_name = parts[1].strip("_")
#     return model_name